In [1]:
# ==========================================
# 1. PREPARACIÓ DE L'ENTORN MODERN
# ==========================================
using Pkg
Pkg.activate("/home/antonibancells/Desktop/Codi/TFM/ProvesModernes")

  Activating project at `~/Desktop/Codi/TFM/ProvesModernes`


In [2]:
using ITensors
using ITensorMPS
using Plots
using LinearAlgebra

Construcción del MPO correspondiente a operador definido mediante una matriz tridiagonal de dimensiones $2^L \times 2^L$ que en la diagonal principal tiene $-\alpha$ excepto su primer elemento, que es $-(\alpha+\gamma)$, y que  en las dos diagonales superior e inferior tiene el valor $\beta$. Para verificar que es correcto, calcularemos su norma teórica o de Frobenius:
$$||\hat{A}||_F=\sqrt{(\alpha+\gamma)^2+(2^L-1)\alpha^2+2\cdot (2^L-1)\beta^2}$$
El resultado calculado mediante esta fórmula tiene que coincidir con el que se obtiene a partir del tensor.
Tambien calcularemos $\langle 00\dots 00|MPO|00\dots00\rangle>$, $\langle 00\dots 00|MPO|00\dots01\rangle>$ y $\langle 00\dots 01|MPO|00\dots 01\rangle>$ para verificar que se obtienen los valores correctos de la matriz.

Una primera posibilidad es la construcción mediante Opsum. La otra posibilidad es la construcción directa de los tensores. 

Opción 1: Construcción automatizada mediante OpSumEsta primera aproximación utiliza el módulo OpSum

Es un método de tipo "caja negra" ideal para definir operadores de forma intuitiva a partir de una lista de términos locales y globales.

¿Cómo funciona internamente?

Acumulación: OpSum almacena todos los términos individuales que se le añaden (en este caso, las interacciones crecen de manera cuadrática con el tamaño del sistema).
Compresión: Internamente genera un Gráfico Acíclico Dirigido (DAG) para encontrar patrones comunes y, finalmente, aplica descomposiciones algebraicas (como QR o SVD) en cada sitio para "comprimir" la red y entregar el MPO final con su dimensión de enlace óptima (D = 3).
Pros y Contras

Ventajas:

Alta flexibilidad: Permite modificar la física o añadir nuevos términos con solo una línea de código.
Seguridad: Elimina por completo el riesgo de cometer errores humanos al gestionar manualmente los canales virtuales de los tensores. Desventajas:
Coste computacional elevado: El uso de memoria RAM y CPU durante la fase de construcción intermedia escala de forma cuadrática o cúbica (O(L²) o O(L³)) debido al procesamiento de cadenas largas de operadores.
⚠️ Nota crítica para QTT:

Aunque el MPO final es óptimo, el proceso de "caja negra" de OpSum puede generar un severo cuello de botella de memoria para sistemas grandes ( 𝐿>20
 ). En simulaciones QTT, donde a menudo se escalan los sitios para representar matrices enormes, este método puede agotar la memoria RAM antes de llegar a la fase de compresión.

In [4]:
#Primera posibilidad
function construir_mpo_qtt_tridiagonal(L::Int, alpha::Float64, beta::Float64, gamma::Float64)
    sites = siteinds("Qubit", L)
    os = OpSum()
    
    # 1. Diagonal homogénea (-4.0) aplicada mediante la Identidad global
    os += -alpha, "Id", 1
    
    # 2. Anomalía en la esquina (1,1) -> Añadimos -1.0 en la rama |000...0><000...0|
    # Construimos un vector plano alternando el nombre del operador y el índice del sitio
    termino_anomalia = Vector{Any}()
    for j in 1:L
        push!(termino_anomalia, "Proj0")
        push!(termino_anomalia, j)
    end
    os += -gamma, termino_anomalia...

    # 3. Bandas secundarias (Superdiagonal y Subdiagonal con Hopping = 2.0)
    for i in 1:L
        # Superdiagonal: S+ en el sitio i, seguido de Proj0 en el resto
        termino_up = Vector{Any}()
        push!(termino_up, "S+")
        push!(termino_up, i)
        for j in (i+1):L
            push!(termino_up, "S-")
            push!(termino_up, j)
        end
        os += beta, termino_up...

        # Subdiagonal: S- en el sitio i, seguido de Proj1 en el resto
        termino_down = Vector{Any}()
        push!(termino_down, "S-")
        push!(termino_down, i)
        for j in (i+1):L
            push!(termino_down, "S+")
            push!(termino_down, j)
        end
        os += beta, termino_down...
    end

    return MPO(os, sites), sites
end

# --- EJEMPLO DE USO PARA TU TFM (Escalado a L = 10 -> Matriz 1024x1024) ---
L = 6
alfa=7.0
beta=3.0
gamma=3.0

MPO_tfm, sites_tfm = construir_mpo_qtt_tridiagonal(L,alfa,beta,gamma)
println("MPO QTT escalable generado con éxito.")
@show maxlinkdim(MPO_tfm)
println("Norma de Frobenius del operador a gran escala: ", norm(MPO_tfm))
norma_teorica = sqrt((alfa+gamma)^2 + (2^L - 1)*(alfa)^2 + 2*(2^L - 1)*(beta)^2)
println("Norma teórica exacta de la matriz clásica:    ", norma_teorica)

MPO QTT escalable generado con éxito.
maxlinkdim(MPO_tfm) = 5
Norma de Frobenius del operador a gran escala: 65.73431371817918
Norma teórica exacta de la matriz clásica:    65.73431371817918


In [5]:
println("\n=== PROTOCOLO DE DIAGNÓSTICO LOCAL (SONDEO DE QUBITS) ===")

# 1. Validar Robin en la esquina (1,1)
prod_robin = [1 for n in 1:L] # Base |000...0>
mps_r = MPS(sites_tfm, prod_robin)
val_r = inner(mps_r', MPO_tfm, mps_r)
println("🔹 Valor esperado en el nodo de frontera (Robin): ", val_r)

# 2. Validar diagonal interna homogénea (2,2)
prod_int = [1 for n in 1:L]; prod_int[end] = 2 # Base |000...1>
mps_i = MPS(sites_tfm, prod_int)
val_i = inner(mps_i', MPO_tfm, mps_i)
println("🔹 Valor esperado en un nodo interno estándar:      ", val_i)

# 3. Validar un salto (Hopping) entre el nodo 1 y el nodo 2
# Calculamos el elemento de matriz <000...01| H |000...00>
val_hop = inner(mps_i', MPO_tfm, mps_r)
println("🔹 Elemento de matriz del salto térmico (Hopping):  ", val_hop)


=== PROTOCOLO DE DIAGNÓSTICO LOCAL (SONDEO DE QUBITS) ===
🔹 Valor esperado en el nodo de frontera (Robin): -10.0
🔹 Valor esperado en un nodo interno estándar:      -7.0
🔹 Elemento de matriz del salto térmico (Hopping):  3.0


In [6]:
function matriz_densa_desde_mpo(mpo, sites)
    L = length(sites)
    N = 2^L
    M = zeros(N, N)
    for col in 0:(N-1)
        ket_bits = [(col >> (L-1-j)) & 1 for j in 0:(L-1)]
        ket = MPS([itensor([1 - ket_bits[j+1], ket_bits[j+1]], sites[j+1]) for j in 0:(L-1)])
        for row in 0:(N-1)
            bra_bits = [(row >> (L-1-j)) & 1 for j in 0:(L-1)]
            bra = MPS([itensor([1 - bra_bits[j+1], bra_bits[j+1]], sites[j+1]) for j in 0:(L-1)])
            M[row+1, col+1] = real(inner(bra', mpo, ket))
        end
    end
    return M
end

# Verificar OpSum
L = 4; α = 7.0; β = 3.0; γ = 3.0
mpo_opsum, sites = construir_mpo_qtt_tridiagonal(L, α, β, γ)

M_opsum = matriz_densa_desde_mpo(mpo_opsum, sites)

println("\nM_opsum (primeras 8×8):")
display(round.(M_opsum[1:8, 1:8], digits=4))


M_opsum (primeras 16×16):


8×8 Matrix{Float64}:
 -10.0   3.0   0.0   0.0   0.0   0.0   0.0   0.0
   3.0  -7.0   3.0   0.0   0.0   0.0   0.0   0.0
   0.0   3.0  -7.0   3.0   0.0   0.0   0.0   0.0
   0.0   0.0   3.0  -7.0   3.0   0.0   0.0   0.0
   0.0   0.0   0.0   3.0  -7.0   3.0   0.0   0.0
   0.0   0.0   0.0   0.0   3.0  -7.0   3.0   0.0
   0.0   0.0   0.0   0.0   0.0   3.0  -7.0   3.0
   0.0   0.0   0.0   0.0   0.0   0.0   3.0  -7.0

**Opción 2:**
**Construcción manual del MPO con condición de contorno (Robin)**
Esta aproximación consiste en programar directamente el autómata o estructura de bloques del MPO. Al introducir una condición de contorno en el primer elemento a través del proyector $P_0$ (que actúa sobre la rama de ceros), la dimensión de enlace óptima del sistema aumenta de $D=3$ a $D=4$.

**¿Cómo funciona internamente?**
Se reservan (allocate) de forma directa los tensores locales fijando la dimensión de enlace $D = 4$ desde el inicio. El operador se construye sitio por sitio mediante matrices cuyos elementos son operadores locales (donde $I$ es la identidad, $S^+$ es el operador de subida, $S^-$ el de bajada y $P_0$ el proyector $|0\rangle\langle0|$):
- Sitio 1 (Vector Fila $1 \times 4$): Inicializa la identidad global, genera los primeros términos de acoplamiento (hopping) y hace nacer en el cuarto canal el término de la condición de contorno modulado por $-\gamma$:$$W_1 = \begin{pmatrix} I & \beta S^+ & \beta S^- & -\gamma P_0 \end{pmatrix}$$
- Sitios intermedios $i$ (Matriz $4 \times 4$): Propaga la identidad y genera nuevos acoplamientos (fila 1), mantiene el acarreo de las cadenas de hopping (filas 2 y 3) y añade un cuarto canal independiente que actúa como un "filtro", propagando el proyector $P_0$ de manera continua a lo largo de todo el tren de espines:$$W_i = \begin{pmatrix} I & \beta S^+ & \beta S^- & 0 \\ 0 & S^- & 0 & 0 \\ 0 & 0 & S^+ & 0 \\ 0 & 0 & 0 & P_0 \end{pmatrix}$$
- Sitio L (Vector Columna $4 \times 1$): Cierra el MPO. La primera componente aplica la diagonal global $-\alpha I$ junto a los últimos hoppings, las componentes 2 y 3 absorben los acarreos binarios y la cuarta componente absorbe el impacto final de la cadena del proyector $P_0$:$$W_L = \begin{pmatrix} -\alpha I + \beta S^+ + \beta S^- \\ S^- \\ S^+ \\ P_0 \end{pmatrix}$$

**Pros y Contras**

*Ventajas:*
- Máxima eficiencia: El coste computacional y el uso de memoria RAM siguen escalando de forma estrictamente lineal ($\mathcal{O}(L)$). No existen pasos intermedios de compresión (SVD/QR) ni sobrecostes ocultos.
- Consumo de memoria mínimo: El pico de memoria durante la construcción es idéntico al tamaño del MPO final, procesando únicamente tensores locales muy pequeños ($4 \times 2 \times 2 \times 4$).

*Desventajas:*
- Baja flexibilidad: Añadir interacciones complejas o más condiciones de contorno requiere rediseñar toda la lógica de los canales virtuales a mano (y aumentar la dimensión de la matriz).
- Riesgo de error humano: Es un método muy propenso a errores de asignación; equivocarse en un solo índice virtual romperá la estructura correlacionada que exige la condición de Robin.

🚀 Ventaja clave para QTT: 
A pesar de haber aumentado la complejidad física del problema añadiendo canales de proyección de largo alcance, la construcción manual garantiza que el tiempo de creación del MPO sea instantáneo incluso para decenas o cientos de qubits ($L > 20$), evitando por completo los cuellos de botella de memoria que sufriría un método automatizado.

In [7]:
# =============================================
# CONSTRUCCIÓN MANUAL CON CONDICIÓN DE CONTORNO
# =============================================
function construir_mpo_manual_perfecto(L::Int, alpha::Float64, beta::Float64, gamma::Float64)
    sites = siteinds("Qubit", L)
    MPO_manual = MPO(sites)
    
    # Índices virtuales de dimensión D = 4
    links = [Index(4, "Link,l=$l") for l in 1:(L-1)]
    
    Id = [1.0 0.0; 0.0 1.0]; P0 = [1.0 0.0; 0.0 0.0]; 
    Sp = [0.0 1.0; 0.0 0.0]; Sm = [0.0 0.0; 1.0 0.0]
    
    # --- SITIO 1: Vector Fila 1x4 ---
    W1 = ITensor(sites[1], prime(sites[1]), links[1])
    for s in 1:2, sp in 1:2
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>1] = Id[s, sp]
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>2] = beta * Sp[s, sp]
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>3] = beta * Sm[s, sp]
        W1[sites[1]=>s, prime(sites[1])=>sp, links[1]=>4] = -gamma * P0[s, sp] # Nace Robin
    end
    MPO_manual[1] = W1
    
    # --- SITIOS INTERMEDIOS: Matrices 4x4 ---
    for i in 2:(L-1)
        W = ITensor(links[i-1], sites[i], prime(sites[i]), links[i])
        for s in 1:2, sp in 1:2
            # Canal 1: Identidad pura (avanza y puede generar nuevos hoppings)
            W[links[i-1]=>1, sites[i]=>s, prime(sites[i])=>sp, links[i]=>1] = Id[s, sp]
            W[links[i-1]=>1, sites[i]=>s, prime(sites[i])=>sp, links[i]=>2] = beta * Sp[s, sp]
            W[links[i-1]=>1, sites[i]=>s, prime(sites[i])=>sp, links[i]=>3] = beta * Sm[s, sp]
            
            # Canal 2: Superdiagonal (Acarreo continuo con Sm según tu código)
            W[links[i-1]=>2, sites[i]=>s, prime(sites[i])=>sp, links[i]=>2] = Sm[s, sp]
            
            # Canal 3: Subdiagonal (Acarreo continuo con Sp según tu código)
            W[links[i-1]=>3, sites[i]=>s, prime(sites[i])=>sp, links[i]=>3] = Sp[s, sp]
            
            # Canal 4: Filtro de Robin (Sigue midiendo P0 en la rama de ceros)
            W[links[i-1]=>4, sites[i]=>s, prime(sites[i])=>sp, links[i]=>4] = P0[s, sp]
        end
        MPO_manual[i] = W
    end
    
    # --- SITIO L: Vector Columna 4x1 ---
    WN = ITensor(links[L-1], sites[L], prime(sites[L]))
    for s in 1:2, sp in 1:2
        # Canal 1: Descarga la diagonal global (-alpha) y añade los hoppings del qubit L
        WN[links[L-1]=>1, sites[L]=>s, prime(sites[L])=>sp] = -alpha * Id[s, sp] + beta * Sp[s, sp] + beta * Sm[s, sp]
        
        # Canal 2 y 3: Reciben el último impacto del acarreo binario de las cadenas largas
        WN[links[L-1]=>2, sites[L]=>s, prime(sites[L])=>sp] = Sm[s, sp]
        WN[links[L-1]=>3, sites[L]=>s, prime(sites[L])=>sp] = Sp[s, sp]
        
        # Canal 4: Cierre estricto de la esquina de Robin
        WN[links[L-1]=>4, sites[L]=>s, prime(sites[L])=>sp] = P0[s, sp]
    end
    MPO_manual[L] = WN
    
    return MPO_manual,sites
end

# --- BATERÍA DE PRUEBA CRUZADA ---
function verificar_todo()
    L = 4  
    alpha = 7.0; beta = 3.0; gamma = 3.0
    
    mpo_manual,sites= construir_mpo_manual_perfecto(L, alpha, beta, gamma)
    
    norma_num = norm(mpo_manual)
    norma_teorica = sqrt((alpha+gamma)^2 + (2^L - 1)*(alpha)^2 + 2*(2^L - 1)*(beta)^2)
    
    println("=================================================================")
    println("   ¡SISTEMA UNIFICADO CON TU CORRECCIÓN DE ACARREO! (L = $L)")
    println("=================================================================")
    println("Norma Teórica Clásica:    ", norma_teorica)
    println("Norma MPO Manual (D=4):   ", norma_num)
    println("Diferencia Absoluta:      ", abs(norma_num - norma_teorica))
    println("=================================================================")
M_manual = matriz_densa_desde_mpo(mpo_manual, sites)

println("\nM_manual (primeras 8×8):")
display(round.(M_manual[1:8, 1:8], digits=4))
end

verificar_todo()

   ¡SISTEMA UNIFICADO CON TU CORRECCIÓN DE ACARREO! (L = 4)
Norma Teórica Clásica:    33.24154027718932
Norma MPO Manual (D=4):   33.24154027718932
Diferencia Absoluta:      0.0

M_manual (primeras 8×8):


8×8 Matrix{Float64}:
 -10.0   3.0   0.0   0.0   0.0   0.0   0.0   0.0
   3.0  -7.0   3.0   0.0   0.0   0.0   0.0   0.0
   0.0   3.0  -7.0   3.0   0.0   0.0   0.0   0.0
   0.0   0.0   3.0  -7.0   3.0   0.0   0.0   0.0
   0.0   0.0   0.0   3.0  -7.0   3.0   0.0   0.0
   0.0   0.0   0.0   0.0   3.0  -7.0   3.0   0.0
   0.0   0.0   0.0   0.0   0.0   3.0  -7.0   3.0
   0.0   0.0   0.0   0.0   0.0   0.0   3.0  -7.0